# Module 5 Homework: BOW, TFIDF, and Vector Spaces

* DS 5001
* Clay Harris: jbm2rt
* Based on the M05 lab notebook by Raf Alvarado

## Overview

This notebook implements a reusable `get_tfidf()` function and uses it to explore term significance in the Austen–Melville corpus at different levels of the OHCO hierarchy. We compare TFIDF rankings at the **book** level versus the **chapter** level and discuss how the choice of "bag" affects which words surface as significant.

---
# 1. Setup

## Config

In [1]:
data_dir = 'HW_5_DATA/'

gradient_cmap = 'YlGnBu'

## OHCO Hierarchy

In [2]:
OHCO  = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
SENTS = OHCO[:4]
PARAS = OHCO[:3]
CHAPS = OHCO[:2]
BOOKS = OHCO[:1]

## Imports

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set()
%matplotlib inline

---
# 2. Load Data

In [4]:
%%time
LIB   = pd.read_csv('LIB.csv').set_index(BOOKS)
TOKEN = pd.read_csv(data_dir + 'TOKEN.csv').set_index(OHCO)
VOCAB = pd.read_csv(data_dir + 'VOCAB.csv').set_index('term_id')

CPU times: user 905 ms, sys: 184 ms, total: 1.09 s
Wall time: 1.15 s


In [5]:
LIB.head()

,book_title,book_file,author,title
book_id,,,,
158,"Emma, by Jane Austen",epubs/AUSTEN_JANE_EMMA-pg158.txt,austen,Emma
946,"Lady Susan, by Jane Austen",epubs/AUSTEN_JANE_LADY_SUSAN-pg946.txt,austen,Lady Susan
1212,"Love And Freindship And Other Early Works, by ...",epubs/AUSTEN_JANE_LOVE_AND_FREINDSHIP_SIC_-pg1...,austen,Love And Freindship And Other Early Works
141,"Mansfield Park, by Jane Austen",epubs/AUSTEN_JANE_MANSFIELD_PARK-pg141.txt,austen,Mansfield Park
121,"Northanger Abbey, by Jane Austen",epubs/AUSTEN_JANE_NORTHANGER_ABBEY-pg121.txt,austen,Northanger Abbey


In [6]:
VOCAB.head()

,term_str,n,num,stop,p_stem
term_id,,,,,
0,NaN,50493,0,0,NaN
1,0,2,1,0,0
2,1,18,1,0,1
3,10,6,1,0,10
4,100,2,1,0,100


In [7]:
TOKEN.head()

pos_tuple  pos  \
book_id chap_num para_num sent_num token_num                               
158     1        1        0        0                ('Emma', 'NNP')  NNP   
                                   1          ('Woodhouse,', 'NNP')  NNP   
                                   2            ('handsome,', 'NN')   NN   
                                   3              ('clever,', 'NN')   NN   
                                   4                  ('and', 'CC')   CC   

                                               token_str   term_str  
book_id chap_num para_num sent_num token_num                         
158     1        1        0        0                Emma       emma  
                                   1          Woodhouse,  woodhouse  
                                   2           handsome,   handsome  
                                   3             clever,     clever  
                                   4                 and        and

---
# 3. Prepare Data

## Drop NAs

In [8]:
VOCAB = VOCAB[~VOCAB.term_str.isna()]
TOKEN = TOKEN[~TOKEN.term_str.isna()]

## Add `term_id` to TOKEN

We map the `term_str` in TOKEN to the integer `term_id` from VOCAB.  
This lets us efficiently join the two tables and build count matrices.

In [9]:
TOKEN['term_id'] = TOKEN.term_str.map(
    VOCAB.reset_index().set_index('term_str').term_id
)
TOKEN.head()

pos_tuple  pos  \
book_id chap_num para_num sent_num token_num                               
158     1        1        0        0                ('Emma', 'NNP')  NNP   
                                   1          ('Woodhouse,', 'NNP')  NNP   
                                   2            ('handsome,', 'NN')   NN   
                                   3              ('clever,', 'NN')   NN   
                                   4                  ('and', 'CC')   CC   

                                               token_str   term_str  term_id  
book_id chap_num para_num sent_num token_num                                  
158     1        1        0        0                Emma       emma    11812  
                                   1          Woodhouse,  woodhouse    40049  
                                   2           handsome,   handsome    16204  
                                   3             clever,     clever     6486  
                                   4                 and        and     1439

## Add `pos_max` to VOCAB

For each vocabulary term, record the most frequent part-of-speech tag observed in TOKEN.  
We use this later to characterise the POS of top-ranked TFIDF terms.

In [10]:
VOCAB['pos_max'] = (
    TOKEN.groupby(['term_id', 'pos']).pos.count()
    .unstack()
    .idxmax(1)
)
VOCAB.head()

,term_str,n,num,stop,p_stem,pos_max
term_id,,,,,,
1,0,2,1,0,0,CD
2,1,18,1,0,1,CD
3,10,6,1,0,10,CD
4,100,2,1,0,100,CD
5,1000,2,1,0,1000,CD


## Add `term_rank` to VOCAB

In [11]:
if 'term_rank' not in VOCAB.columns:
    VOCAB = VOCAB.sort_values('n', ascending=False).reset_index()
    VOCAB.index.name = 'term_rank'
    VOCAB = VOCAB.reset_index()
    VOCAB = VOCAB.set_index('term_id')
    VOCAB['term_rank'] = VOCAB['term_rank'] + 1

VOCAB.head()

,term_rank,term_str,n,num,stop,p_stem,pos_max
term_id,,,,,,,
35407,1,the,110093,0,1,the,DT
24365,2,of,65993,0,1,of,IN
1439,3,and,63528,0,1,and,CC
35891,4,to,56220,0,1,to,TO
199,5,a,44566,0,1,a,DT


---
# 4. The `get_tfidf()` Function

The function below computes a TFIDF matrix from any TOKEN dataframe.  
It accepts five arguments:

| Argument | Type | Description |
|---|---|---|
| `TOKEN` | DataFrame | The token table (must contain `term_id` and `pos` columns). |
| `bag` | list | OHCO level to use as the document unit (e.g. `BOOKS`, `CHAPS`). |
| `count_method` | str | `'n'` = raw token count; `'c'` = binary (presence/absence). |
| `tf_method` | str | TF normalisation: `'sum'`, `'max'`, `'log'`, `'double_norm'`, `'raw'`, `'binary'`. |
| `idf_method` | str | IDF formula: `'standard'`, `'max'`, `'smooth'`. |

It returns the TFIDF matrix (documents × terms).

In [12]:
def get_tfidf(TOKEN, bag, count_method='n', tf_method='sum', idf_method='standard', tf_norm_k=0.5):
    """
    Compute a TFIDF matrix from a TOKEN dataframe.

    Parameters
    ----------
    TOKEN        : pd.DataFrame   Token table indexed by OHCO levels; must have 'term_id' column.
    bag          : list           OHCO prefix defining the document unit.
    count_method : str            'n' for token count, 'c' for binary count.
    tf_method    : str            TF weighting method.
    idf_method   : str            IDF weighting method.
    tf_norm_k    : float          Smoothing constant for double_norm (default 0.5).

    Returns
    -------
    TFIDF : pd.DataFrame   TFIDF matrix (docs × terms).
    """

    # ── Step 1: Bag of Words ──────────────────────────────────────────────────
    BOW = (
        TOKEN.groupby(bag + ['term_id'])
        .term_id.count()
        .to_frame()
        .rename(columns={'term_id': 'n'})
    )
    BOW['c'] = BOW.n.astype('bool').astype('int')   # binary count

    # ── Step 2: Document-Term Count Matrix ───────────────────────────────────
    DTCM = BOW[count_method].unstack().fillna(0).astype('int')

    # ── Step 3: Term Frequency ────────────────────────────────────────────────
    if tf_method == 'sum':
        TF = DTCM.T / DTCM.T.sum()
    elif tf_method == 'max':
        TF = DTCM.T / DTCM.T.max()
    elif tf_method == 'log':
        TF = np.log10(1 + DTCM.T)
    elif tf_method == 'raw':
        TF = DTCM.T
    elif tf_method == 'double_norm':
        TF = DTCM.T / DTCM.T.max()
        TF = tf_norm_k + (1 - tf_norm_k) * TF[TF > 0]
    elif tf_method == 'binary':
        TF = DTCM.T.astype('bool').astype('int')
    else:
        raise ValueError(f"Unknown tf_method: '{tf_method}'")
    TF = TF.T

    # ── Step 4: Document Frequency & Inverse Document Frequency ──────────────
    DF = DTCM[DTCM > 0].count()
    N  = DTCM.shape[0]          # number of documents at this bag level

    if idf_method == 'standard':
        IDF = np.log10(N / DF)
    elif idf_method == 'max':
        IDF = np.log10(DF.max() / DF)
    elif idf_method == 'smooth':
        IDF = np.log10((1 + N) / (1 + DF)) + 1
    else:
        raise ValueError(f"Unknown idf_method: '{idf_method}'")

    # ── Step 5: TFIDF ─────────────────────────────────────────────────────────
    TFIDF = TF * IDF

    return TFIDF

---
# 5. Book-Level TFIDF  (bag = BOOKS, count_method = 'n')

Here each **book** is a document.  
With only 20 books in the corpus, IDF discriminates coarsely — terms that appear across many books approach zero, while terms concentrated in a single book score high.

In [13]:
%%time
TFIDF_BOOKS = get_tfidf(
    TOKEN,
    bag          = BOOKS,
    count_method = 'n',
    tf_method    = 'sum',
    idf_method   = 'standard'
)
print(f"TFIDF shape (book level): {TFIDF_BOOKS.shape}")
TFIDF_BOOKS.head()

TFIDF shape (book level): (20, 40478)
CPU times: user 2.82 s, sys: 85.8 ms, total: 2.91 s
Wall time: 2.97 s


term_id,1,2,3,4,5,6,7,8,9,10,...,40471,40472,40473,40474,40475,40476,40477,40478,40479,40480
book_id,,,,,,,,,,,,,,,,,,,,,
105,0.0,0.000013,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
121,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
141,0.0,0.000000,0.000013,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000008
158,0.0,0.000000,0.000000,0.0,0.0,0.000012,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
161,0.0,0.000004,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


## Top 20 Words by TFIDF Sum – Book Level

We sum each term's TFIDF score across all books (documents) and take the top 20.

In [14]:
# Sum TFIDF across books for each term
tfidf_sum_books = TFIDF_BOOKS.sum().rename('tfidf_sum')

# Join to VOCAB to get term_str and pos_max
top20_books = (
    VOCAB[['term_str', 'pos_max', 'term_rank', 'n']]
    .join(tfidf_sum_books)
    .sort_values('tfidf_sum', ascending=False)
    .head(20)
)

top20_books.style.background_gradient(cmap=gradient_cmap)

,term_str,pos_max,term_rank,n,tfidf_sum
term_id,,,,,
26302,pierre,NNP,149,1525,0.009838
11648,elinor,NNP,330,623,0.006763
19306,israel,NNP,385,519,0.006504
38673,vernon,NNP,1841,104,0.005857
2644,babbalanja,NNP,370,547,0.005394
22176,media,NNP,401,497,0.004940
5540,catherine,NNP,364,557,0.004324
21823,marianne,NNP,398,499,0.004316
29073,reginald,NNP,2490,74,0.004167


---
# 6. Chapter-Level TFIDF  (bag = CHAPS, count_method = 'n')

Now each **chapter** is a document.  
With hundreds of chapters, IDF can make finer distinctions — a word that varies chapter-to-chapter (e.g., a plot-specific noun) will score differently than at the book level.

In [15]:
%%time
TFIDF_CHAPS = get_tfidf(
    TOKEN,
    bag          = CHAPS,
    count_method = 'n',
    tf_method    = 'sum',
    idf_method   = 'standard'
)
print(f"TFIDF shape (chapter level): {TFIDF_CHAPS.shape}")
TFIDF_CHAPS.head()

TFIDF shape (chapter level): (1122, 40478)
CPU times: user 3.58 s, sys: 805 ms, total: 4.39 s
Wall time: 3.93 s


term_id           1        2      3      4      5      6      7      8      \
book_id chap_num                                                             
105     1           0.0  0.00153    0.0    0.0    0.0    0.0    0.0    0.0   
        2           0.0  0.00000    0.0    0.0    0.0    0.0    0.0    0.0   
        3           0.0  0.00000    0.0    0.0    0.0    0.0    0.0    0.0   
        4           0.0  0.00000    0.0    0.0    0.0    0.0    0.0    0.0   
        5           0.0  0.00000    0.0    0.0    0.0    0.0    0.0    0.0   

term_id           9      10     ...  40471  40472  40473  40474  40475  40476  \
book_id chap_num                ...                                             
105     1           0.0    0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0   
        2           0.0    0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0   
        3           0.0    0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0   
        4           0.0    0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0   
        5           0.0    0.0  ...    0.0    0.0    0.0    0.0    0.0    0.0   

term_id           40477  40478  40479  40480  
book_id chap_num                              
105     1           0.0    0.0    0.0    0.0  
        2           0.0    0.0    0.0    0.0  
        3           0.0    0.0    0.0    0.0  
        4           0.0    0.0    0.0    0.0  
        5           0.0    0.0    0.0    0.0  

[5 rows x 40478 columns]

## Top 20 Words by TFIDF Sum – Chapter Level

In [16]:
# Sum TFIDF across chapters for each term
tfidf_sum_chaps = TFIDF_CHAPS.sum().rename('tfidf_sum')

top20_chaps = (
    VOCAB[['term_str', 'pos_max', 'term_rank', 'n']]
    .join(tfidf_sum_chaps)
    .sort_values('tfidf_sum', ascending=False)
    .head(20)
)

top20_chaps.style.background_gradient(cmap=gradient_cmap)

,term_str,pos_max,term_rank,n,tfidf_sum
term_id,,,,,
31648,she,PRP,24,12153,1.349173
16730,her,PRP$,15,17020,1.331294
26302,pierre,NNP,149,1525,1.154340
40387,you,PRP,20,14466,0.756695
23260,mr,NNP,75,3420,0.706684
17566,i,PRP,7,27810,0.664376
39540,whale,NN,193,1180,0.594694
23261,mrs,NNP,99,2664,0.593591
35574,thou,NN,233,916,0.586061


---
# 7. Comparison and Analysis

## Side-by-side: Book vs Chapter

In [17]:
compare = pd.DataFrame({
    'book_top20':  top20_books.term_str.values,
    'book_pos':    top20_books.pos_max.values,
    'chap_top20':  top20_chaps.term_str.values,
    'chap_pos':    top20_chaps.pos_max.values,
}, index=range(1, 21))
compare.index.name = 'rank'
compare

,book_top20,book_pos,chap_top20,chap_pos
rank,,,,
1,pierre,NNP,she,PRP
2,elinor,NNP,her,PRP$
3,israel,NNP,pierre,NNP
4,vernon,NNP,you,PRP
5,babbalanja,NNP,mr,NNP
6,media,NNP,i,PRP
7,catherine,NNP,whale,NN
8,marianne,NNP,mrs,NNP
9,reginald,NNP,thou,NN


In [18]:
set_books = set(top20_books.term_str)
set_chaps = set(top20_chaps.term_str)

shared    = set_books & set_chaps
only_book = set_books - set_chaps
only_chap = set_chaps - set_books

print(f"Shared terms        : {sorted(shared)}")
print(f"Book-only terms     : {sorted(only_book)}")
print(f"Chapter-only terms  : {sorted(only_chap)}")

Shared terms        : ['babbalanja', 'media', 'pierre']
Book-only terms     : ['catherine', 'crawford', 'darcy', 'elinor', 'elliot', 'emma', 'fanny', 'frederica', 'israel', 'marianne', 'mohi', 'reginald', 'tilney', 'vernon', 'wentworth', 'weston', 'yoomy']
Chapter-only terms  : ['captain', 'her', 'i', 'isabel', 'me', 'miss', 'mr', 'mrs', 'my', 'she', 'sir', 'thee', 'thou', 'um', 'whale', 'you', 'your']


In [19]:
print("Book top-20 POS distribution:")
print(top20_books.pos_max.value_counts().to_string())
print()
print("Chapter top-20 POS distribution:")
print(top20_chaps.pos_max.value_counts().to_string())

Book top-20 POS distribution:
pos_max
NNP    20

Chapter top-20 POS distribution:
pos_max
NNP     9
PRP     4
PRP$    3
NN      3
JJ      1


---
## Q1 — Top 20 Words at Book Level Using the `'n'` Count Method

All 20 terms are **proper nouns (NNP)** — exclusively character names bound to a single book: *pierre*, *elinor*, *israel*, *babbalanja*, *catherine*, *darcy*, *emma*, etc.

This is exactly what book-level IDF selects for. With only 20 documents, any term that appears in all 20 books gets IDF = log(20/20) = 0 and is eliminated. A character name that belongs to exactly one book gets IDF = log(20/1) ≈ 1.30 — the maximum — and its high within-book TF pushes it to the top. Common words are automatically suppressed; character names are automatically surfaced.

---
## Q2 — Book vs Chapter: Are the Top 20 Words the Same? How Do They Differ by POS?

The lists are almost entirely different. Only 3 terms are shared: **pierre**, **babbalanja**, and **media** — all major characters from Melville's *Mardi*, frequent enough within chapters to survive both bag levels.

**Book level: 100% NNP.** Every entry is a character name exclusive to one book.

**Chapter level: a mix of PRP, NNP, and NN.**  
The most striking change is the appearance of **pronouns** (*she*, *her*, *you*, *I*, *me*, *my*, *your*) at the top of the chapter list — none of these appear in the book list. At book level, pronouns appear in all 20 books so their IDF collapses to zero. At the chapter level, however, many of Melville's sea-action chapters contain few or no female pronouns, meaning *she* and *her* are absent from a meaningful fraction of chapters and earn a non-trivial IDF. Combined with their very high TF in Austen chapters, they rise to the top.